# Transfer Learning for Computer Vision — Caltech-101

Step-by-step version of `4-transfer_101.py`: a pretrained **MobileNetV2**
is used as a feature extractor, a new classification head is trained on
Caltech-101 (102 classes = 101 objects + background), then the top layers
of the backbone are fine-tuned. The final model is saved as
`caltech101_model.h5` (target: ≥ 85% validation accuracy).

> **Runtime → Change runtime type → GPU** before running.

## 0. Setup

In [ ]:
!pip install tensorflow

Install `gdown` to be able to download the dataset from Google Drive.

In [ ]:
!pip install gdown

Download and unzip the `caltech-101.zip` file using the provided Google Drive
link. The dataset is expected to be extracted into a folder named
`101_ObjectCategories`.

In [ ]:
import gdown
import os
import shutil

# Google Drive ID for caltech-101.zip
drive_file_id = '1ci0p73yJ-ckdirRtgCAdjf53y2uPSaJN'
output_zip_file = 'caltech-101.zip'

# Download the file
gdown.download(f'https://drive.google.com/uc?id={drive_file_id}', output_zip_file, quiet=False)

# Unzip the main file, which creates a directory named 'caltech-101'
!unzip -q {output_zip_file} -d .

# Untar 101_ObjectCategories.tar.gz inside 'caltech-101'
extracted_dir = 'caltech-101'
os.chdir(extracted_dir)
!tar -xzf 101_ObjectCategories.tar.gz
os.chdir('..')

# Move '101_ObjectCategories' to the working directory (DATA_DIR below)
shutil.move(os.path.join(extracted_dir, '101_ObjectCategories'), '.')

# Clean up the intermediate directory and zip file
shutil.rmtree(extracted_dir)
os.remove(output_zip_file)

if os.path.exists('101_ObjectCategories'):
    print("Successfully downloaded and extracted '101_ObjectCategories'.")
else:
    print("Error: '101_ObjectCategories' directory not found after processing.")

### Imports and configuration

Same constants as the top of `4-transfer_101.py`.

In [ ]:
import tensorflow as tf
from tensorflow import keras

DATA_DIR = "101_ObjectCategories"
MODEL_PATH = "caltech101_model.h5"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS_HEAD = 10
EPOCHS_FINETUNE = 15
UNFREEZE_LAYERS = 30
WEIGHTS = "imagenet"

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 1. Load and prepare the datasets

`image_dataset_from_directory` builds an 80/20 train/validation split
(same `seed` for both subsets, so they don't overlap). Labels are integers,
which pairs with `sparse_categorical_crossentropy` later. The number of
classes is read from the folder names (102, including `BACKGROUND_Google`).

In [ ]:
train_ds, val_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="both", seed=42,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="int")
num_classes = len(train_ds.class_names)  # 101 objects + background
print("Number of classes:", num_classes)

## 2. Data augmentation + preprocessing

Augmentation (flip, rotation, zoom, contrast) is applied **only to the
training set**. Both sets then go through
`mobilenet_v2.preprocess_input`, which scales pixels from `[0, 255]` to
`[-1, 1]` — the range MobileNetV2 was trained on.

Doing this in the `tf.data` pipeline (instead of inside the model) keeps the
saved `.h5` model free of custom layers, so it loads without extra code.
The trade-off: raw images must be passed through `preprocess_input` before
calling the saved model.

In [ ]:
def build_data_augmentation():
    """ builds a seeded Sequential model of image augmentation layers """
    return keras.Sequential([
        keras.layers.RandomFlip("horizontal", seed=42),
        keras.layers.RandomRotation(0.1, seed=42),
        keras.layers.RandomZoom(0.1, seed=42),
        keras.layers.RandomContrast(0.1, seed=42),
    ])


augment = build_data_augmentation()
preprocess = keras.applications.mobilenet_v2.preprocess_input
train_ds = train_ds.map(
    lambda x, y: (preprocess(augment(x, training=True)), y),
    num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.map(
    lambda x, y: (preprocess(x), y),
    num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

## 3. Build the model (frozen base)

MobileNetV2 with ImageNet weights and `include_top=False` (no 1000-class
ImageNet head). The base is frozen and called with `training=False`, so its
BatchNorm layers stay in inference mode. On top: global average pooling →
dropout → a new softmax layer with one output per Caltech-101 class.

In [ ]:
def build_model(num_classes):
    """ builds a frozen MobileNetV2 base with a new softmax head """
    base_model = keras.applications.MobileNetV2(
        weights=WEIGHTS, input_shape=IMG_SIZE + (3,), include_top=False)
    base_model.trainable = False

    inputs = keras.Input(shape=IMG_SIZE + (3,))
    x = base_model(inputs, training=False)  # keep BatchNorm in inference
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs), base_model


model, base_model = build_model(num_classes)
model.summary()

## 4. Phase 1 — train the classification head only

Only the new head is trainable. `EarlyStopping` restores the best weights
and `ReduceLROnPlateau` lowers the learning rate when validation loss stalls.

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=3, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.2, patience=2),
]

model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
history_head = model.fit(train_ds, validation_data=val_ds,
                         epochs=EPOCHS_HEAD, callbacks=callbacks)

## 5. Phase 2 — fine-tune the top layers of the base model

The last `UNFREEZE_LAYERS` layers of MobileNetV2 are unfrozen, except
BatchNormalization layers: their running statistics were learned on
ImageNet and updating them on a small dataset tends to hurt accuracy.
The model is recompiled (required for the change to take effect) with a
100x smaller learning rate so the pretrained weights are only nudged.

In [ ]:
def unfreeze_top(base_model, n_layers):
    """ unfreezes the last n_layers of base_model, BatchNorm stays frozen """
    base_model.trainable = True
    split = max(len(base_model.layers) - n_layers, 0)
    for i, layer in enumerate(base_model.layers):
        is_bn = isinstance(layer, keras.layers.BatchNormalization)
        layer.trainable = i >= split and not is_bn


unfreeze_top(base_model, UNFREEZE_LAYERS)
print("Trainable layers in base:",
      sum(layer.trainable for layer in base_model.layers))

model.compile(optimizer=keras.optimizers.Adam(1e-5),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
history_finetune = model.fit(train_ds, validation_data=val_ds,
                             epochs=EPOCHS_FINETUNE, callbacks=callbacks)

## 6. Evaluate and save

Because `EarlyStopping` uses `restore_best_weights=True`, the model in memory
holds its best weights, so a single save at the end is enough.

In [ ]:
_, val_acc = model.evaluate(val_ds)
print("Validation accuracy: {:.4f}".format(val_acc))
model.save(MODEL_PATH)
print("Model saved as", MODEL_PATH)

## 7. The whole pipeline as `train_transfer_model()`

The task asks for a function `train_transfer_model()`. This cell wraps
sections 1–6 into that function, reusing `build_data_augmentation()`,
`build_model()` and `unfreeze_top()` defined above. It matches
`train_transfer_model()` in `4-transfer_101.py`.

Running this cell only **defines** the function. Uncomment the last line to
train end to end in one call. That repeats the training already done in
sections 1–6, so it isn't needed if you ran those.

In [ ]:
def load_datasets():
    """ loads Caltech-101 as preprocessed train / validation datasets """
    train_ds, val_ds = keras.utils.image_dataset_from_directory(
        DATA_DIR, validation_split=0.2, subset="both", seed=42,
        image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="int")
    num_classes = len(train_ds.class_names)  # 101 objects + background

    augment = build_data_augmentation()
    preprocess = keras.applications.mobilenet_v2.preprocess_input
    train_ds = train_ds.map(
        lambda x, y: (preprocess(augment(x, training=True)), y),
        num_parallel_calls=tf.data.AUTOTUNE)
    val_ds = val_ds.map(
        lambda x, y: (preprocess(x), y),
        num_parallel_calls=tf.data.AUTOTUNE)
    train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
    val_ds = val_ds.prefetch(tf.data.AUTOTUNE)
    return train_ds, val_ds, num_classes


def train_transfer_model():
    """ trains in two phases (head, then fine-tuning) and saves the model """
    train_ds, val_ds, num_classes = load_datasets()
    model, base_model = build_model(num_classes)
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_accuracy", patience=3, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.2, patience=2),
    ]

    # Phase 1: train the classification head only
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD,
              callbacks=callbacks)

    # Phase 2: fine-tune the top layers with a small learning rate
    unfreeze_top(base_model, UNFREEZE_LAYERS)
    model.compile(optimizer=keras.optimizers.Adam(1e-5),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FINETUNE,
              callbacks=callbacks)

    _, val_acc = model.evaluate(val_ds)
    print("Validation accuracy: {:.4f}".format(val_acc))
    model.save(MODEL_PATH)
    return model


# model = train_transfer_model()

### Verify the saved model

In [ ]:
import os

if os.path.exists(MODEL_PATH):
    print(f"The trained model '{MODEL_PATH}' was successfully saved.")
else:
    print(f"Error: the trained model '{MODEL_PATH}' was not found.")

!ls -F